# Chart Builder
Runs the chart builder independently (requires scorecard output).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if Path.cwd().name == 'certification_framework' else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import json
from scripts.computation.scorecard_builder import build_from_file as build_scorecard
from scripts.computation.chart_builder import build_from_file as build_charts

NB_DATA = ROOT / 'notebooks' / 'data'
phase1_path = NB_DATA / 'parsed_context.json'

# Build scorecard first (charts depend on it)
scorecard_result = build_scorecard(phase1_path)
scorecard_dims = scorecard_result['scorecard']['dimensions']

# Build charts
result = build_charts(phase1_path, scorecard_dims, render=False)

# Save to notebook-local data
(NB_DATA / 'charts.json').write_text(
    json.dumps(result, indent=2, default=str, ensure_ascii=False), encoding='utf-8'
)

print(f'=== Charts ({len(result["charts"])}) ===')
for name, chart in result['charts'].items():
    print(f'  {name:25s} type={chart["chart_type"]}')

=== Charts (9) ===
  scorecard_radar           type=radar
  ttd_bar                   type=grouped_bar
  ttm_bar                   type=grouped_bar
  rates_bar                 type=grouped_bar
  accuracy_heatmap          type=heatmap
  reasoning_bar             type=grouped_bar
  hallucination_bar         type=grouped_bar
  compliance_bar            type=grouped_bar
  token_stacked             type=stacked_bar


In [3]:
print('=== Validation: Charts ===')
all_ok = True

expected_charts = ['scorecard_radar', 'ttd_bar', 'ttm_bar', 'rates_bar',
                   'accuracy_heatmap', 'reasoning_bar', 'hallucination_bar',
                   'compliance_bar', 'token_stacked']
for name in expected_charts:
    ok = 'PASS' if name in result['charts'] else 'FAIL'
    if ok == 'FAIL': all_ok = False
    print(f'  {ok} chart present: {name}')

# All charts must have chart_type
for name, chart in result['charts'].items():
    ok = 'PASS' if 'chart_type' in chart else 'FAIL'
    if ok == 'FAIL': all_ok = False
    print(f'  {ok} {name} has chart_type: {chart.get("chart_type", "MISSING")}')

print(f'\nResult: {"ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED"}')

=== Validation: Charts ===
  PASS chart present: scorecard_radar
  PASS chart present: ttd_bar
  PASS chart present: ttm_bar
  PASS chart present: rates_bar
  PASS chart present: accuracy_heatmap
  PASS chart present: reasoning_bar
  PASS chart present: hallucination_bar
  PASS chart present: compliance_bar
  PASS chart present: token_stacked
  PASS scorecard_radar has chart_type: radar
  PASS ttd_bar has chart_type: grouped_bar
  PASS ttm_bar has chart_type: grouped_bar
  PASS rates_bar has chart_type: grouped_bar
  PASS accuracy_heatmap has chart_type: heatmap
  PASS reasoning_bar has chart_type: grouped_bar
  PASS hallucination_bar has chart_type: grouped_bar
  PASS compliance_bar has chart_type: grouped_bar
  PASS token_stacked has chart_type: stacked_bar

Result: ALL CHECKS PASSED


## Rendered Charts

In [11]:
import importlib
import scripts.computation.chart_renderer as _cr
importlib.reload(_cr)
from scripts.computation.chart_renderer import create_figure

for name, chart in result['charts'].items():
    fig = create_figure(chart)
    fig.show()